In [2]:
#@markdown <br><center><img src='https://1.1.1.1/media/warp-desktop.png' height="200" alt="Wrap+"/></center>
#@markdown <center><h3><b>WARP (1.1.1.1)</i></b></h3></center><br><hr><br>

import httpx
import json
import random
import string
import time
from datetime import datetime
import logging
import os

# Configuration
WARP_CLIENT_ID = "594d7d3d-4347-4d1c-aa07-58a1a49ab58a"  # @param {type:"string"}
LOG_LEVEL = "INFO"  # @param ["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"]
USE_PROXY = False  # @param {type:"boolean"}
PROXY_URL = "http://your-proxy-url:port"  # @param {type:"string"}  # e.g., "http://127.0.0.1:8080"
REQUEST_TIMEOUT = 10  # in seconds
COOLDOWN_RANGE_MIN = 60  # Minimum cooldown time in seconds
COOLDOWN_RANGE_MAX = 120  # Maximum cooldown time in seconds
MAX_RETRIES = 3 # Maximum number of retries for failed requests

# Logging setup
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.StreamHandler()  # Output to console
    ]
)
logger = logging.getLogger(__name__)

def gen_string(length):
    """Generates a random string of specified length."""
    try:
        letters = string.ascii_letters + string.digits
        return ''.join(random.choice(letters) for _ in range(length))
    except Exception as e:
        logger.error(f"Error in gen_string: {e}")
        return None

def digit_string(length):
    """Generates a random string of digits."""
    try:
        digit = string.digits
        return ''.join(random.choice(digit) for _ in range(length))
    except Exception as e:
        logger.error(f"Error in digit_string: {e}")
        return None

def send_request(url, data, headers, proxies=None):
    """Sends an HTTP POST request with retry logic."""
    for attempt in range(MAX_RETRIES):
        try:
            response = httpx.post(url, data=data, headers=headers, proxies=proxies, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()  # Raise an exception for bad status codes (4xx or 5xx)
            return response
        except httpx.RequestError as e:
            logger.error(f"Request error (attempt {attempt + 1}/{MAX_RETRIES}): {e}")
            if attempt < MAX_RETRIES - 1:
                retry_after = random.randint(COOLDOWN_RANGE_MIN, COOLDOWN_RANGE_MAX)
                logger.info(f"Retrying after {retry_after} seconds...")
                time.sleep(retry_after)
            else:
                raise  # Re-raise the last exception if all retries failed
        except httpx.HTTPStatusError as e:
            logger.error(f"HTTP error (attempt {attempt + 1}/{MAX_RETRIES}): {e}, Status Code: {e.response.status_code}")
            return e.response # Return the response to be handled by the caller
        except Exception as e:
            logger.error(f"An unexpected error occurred (attempt {attempt + 1}/{MAX_RETRIES}): {e}")
            if attempt < MAX_RETRIES - 1:
                retry_after = random.randint(COOLDOWN_RANGE_MIN, COOLDOWN_RANGE_MAX)
                logger.info(f"Retrying after {retry_after} seconds...")
                time.sleep(retry_after)
            else:
                raise  # Re-raise the last exception if all retries failed
    return None #This should never be reached

def main():
    """Main function to run the script."""
    global SUCCESS_COUNT, FAIL_COUNT, USE_PROXY  # Access the global counters

    if USE_PROXY and not PROXY_URL:
        logger.warning("USE_PROXY is True but PROXY_URL is not set.  Not using a proxy.")
        USE_PROXY = False

    proxies = {"http": PROXY_URL, "https": PROXY_URL} if USE_PROXY else None
    if USE_PROXY:
        logger.info(f"Using proxy: {PROXY_URL}")

    while True:
        try:
            install_id = gen_string(22)
            if install_id is None:
                logger.error("Failed to generate install_id. Skipping this attempt.")
                FAIL_COUNT += 1
                continue

            key = gen_string(43)
            if key is None:
                logger.error("Failed to generate key. Skipping this attempt.")
                FAIL_COUNT += 1
                continue
            fcm_token = gen_string(134)
            if fcm_token is None:
                logger.error("Failed to generate fcm_token. Skipping this attempt")
                FAIL_COUNT += 1
                continue
            body = {
                "key": f"{key}=",
                "install_id": install_id,
                "fcm_token": f"{install_id}:APA91b{fcm_token}",
                "referrer": WARP_CLIENT_ID,
                "warp_enabled": False,
                "tos": f"{datetime.now().isoformat()[:-3]}+02:00",
                "type": "Android",  # Consider making this configurable
                "locale": "es_ES",  # Consider making this configurable
            }
            data = json.dumps(body).encode("utf8")
            url = f"https://api.cloudflareclient.com/v0a{digit_string(3)}/reg"
            headers = {
                "Content-Type": "application/json; charset=UTF-8",
                "Host": "api.cloudflareclient.com",
                "Connection": "Keep-Alive",
                "Accept-Encoding": "gzip",
                "User-Agent": "okhttp/3.12.1",  # Consider making this configurable
            }

            response = send_request(url, data, headers, proxies)
            if response: # response is not None
                if response.status_code == 200:
                    SUCCESS_COUNT += 1
                    logger.info(f"PASSED: +1GB (total: {SUCCESS_COUNT}GB, failed: {FAIL_COUNT})")
                else:
                    FAIL_COUNT += 1
                    logger.warning(f"FAILED: {response.status_code}, Content: {response.text}")
            else:
                FAIL_COUNT += 1
                logger.error(f"FAILED: Request failed after {MAX_RETRIES} attempts.")

        except Exception as e:
            logger.critical(f"An unexpected error occurred in main loop: {e}")

        # Cooldown
        cooldown_time = random.randint(COOLDOWN_RANGE_MIN, COOLDOWN_RANGE_MAX)
        logger.info(f"Sleeping for {cooldown_time} seconds.")
        time.sleep(cooldown_time)


if __name__ == "__main__":
    main()



ERROR:__main__:An unexpected error occurred (attempt 1/3): post() got an unexpected keyword argument 'proxies'


KeyboardInterrupt: 